# 🚀 PHẦN 1: DOCTR GPU INFERENCE TRÊN DOCUNET BENCHMARK

Notebook này chuyên biệt cho nhiệm vụ: **Chạy mô hình DocTr nắn phẳng toàn bộ 130 ảnh DocUNet bằng GPU T4 trên Google Colab** (~20 giây).

> **Yêu cầu quan trọng:**
> 1. Vào **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** -> Bấm **Save**.
> 2. Trên Google Drive của bạn cần có sẵn thư mục `Document_Dewarp_Benchmark` chứa `model_pretrained` và `data_benchmark` (xem hướng dẫn trong file `HUONG_DAN_CHAY_BENCHMARK_COLAB.md`).

### Bước 1: Kiểm tra GPU & Kết nối Google Drive

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

### Bước 2: Clone mã nguồn từ GitHub & Cài đặt thư viện

In [ ]:
import os
# Đổi sang URL GitHub repo của bạn
GITHUB_REPO_URL = 'https://github.com/nck345/test_doctr.git'

if not os.path.exists('/content/DocTr-Plus'):
    !git clone {GITHUB_REPO_URL} /content/DocTr-Plus
else:
    !cd /content/DocTr-Plus && git pull

%cd /content/DocTr-Plus
!pip install -q timm opencv-python pillow scikit-image tqdm


### Bước 3: Thiết lập danh sách Benchmark cần nắn phẳng
Hỗ trợ chạy 1 tập đơn lẻ (ví dụ `'docunet_paper_subset'`), danh sách nhiều tập (ví dụ `['docunet_paper_subset', 'dir300', 'docreal']`) hoặc toàn bộ các tập (`'all'`).

In [ ]:
import os

DRIVE_DIR = '/content/drive/MyDrive/Document_Dewarp_Benchmark'

# ==============================================================================
# CHỌN TẬP BENCHMARK ĐỂ CHẠY NẮN PHẲNG:
#    - Một tập đơn:  'docunet_paper_subset' (hoặc 'docunet_130', 'dir300', 'docreal')
#    - Nhiều tập:    ['docunet_paper_subset', 'dir300', 'docreal']
#    - Toàn bộ:      'all'
# ==============================================================================
BENCHMARKS = 'all'  # Hoặc ['docunet_paper_subset', 'dir300', 'docreal'] hoặc 'docunet_paper_subset'

ALL_BENCHMARKS = ['docunet_paper_subset', 'docunet_130', 'dir300', 'docreal']
if BENCHMARKS == 'all':
    target_benchmarks = ALL_BENCHMARKS
elif isinstance(BENCHMARKS, str):
    target_benchmarks = [BENCHMARKS]
else:
    target_benchmarks = list(BENCHMARKS)

CHECKPOINT_PATH = f'{DRIVE_DIR}/model_pretrained/DocTrP.pth'

print('=' * 75)
print(f'Tổng số tập benchmark cần xử lý: {len(target_benchmarks)}')
for bm in target_benchmarks:
    c_dir = f'{DRIVE_DIR}/benchmarks/{bm}/crop'
    if not os.path.exists(c_dir):
        c_dir = f'{DRIVE_DIR}/data_benchmark/crop/crop'
    print(f'- {bm:<22} | Input: {c_dir} (Tồn tại: {os.path.exists(c_dir)})')
print('Checkpoint path :', CHECKPOINT_PATH, '- Tồn tại:', os.path.exists(CHECKPOINT_PATH))
print('=' * 75)


### Bước 4: Chạy nắn phẳng bằng GPU T4
Mô hình nạp trọng số 1 lần duy nhất vào VRAM và tự động xử lý tuần tự toàn bộ các tập benchmark đã chọn.

In [ ]:
import os, time

bm_args = ' '.join(target_benchmarks)
print(f'🚀 Bắt đầu chạy nắn phẳng DocTr cho {len(target_benchmarks)} tập: {bm_args}\n')
t_start = time.time()

!python inference.py --base_dir '{DRIVE_DIR}' --benchmarks {bm_args} --GeoTr_path '{CHECKPOINT_PATH}'

total_time = time.time() - t_start
print('\n' + '=' * 75)
print(f'🎉 TỔNG KẾT: Hoàn tất nắn phẳng {len(target_benchmarks)} tập trong {total_time:.1f} giây')
for bm in target_benchmarks:
    p_dir = f'{DRIVE_DIR}/predictions/DocTr/{bm}'
    n_done = len([f for f in os.listdir(p_dir) if f.endswith('.png')]) if os.path.exists(p_dir) else 0
    print(f'- Tập {bm:<22}: {n_done} ảnh tại {p_dir}')
print('=' * 75)
